# Voice search

Speech query to a news or product search. Import the pipeline, load the models, check intent and query extraction, then open the Gradio demo.


## Imports

Load the record, pipeline stages, logging, and environment helper.


In [1]:
from src.utils.voice_search_model import VoiceSearch
from src.utils.transcribe import Transcriber
from src.utils.intent import IntentDetector
from src.utils.query import QueryExtractor
from src.utils.search import Searcher
from src.utils.logging_setup import setup_logging
from src.app.gradio_demo import build_demo
from src.app.pipeline import VoiceSearchPipeline
from dotenv import load_dotenv

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Environment

Read `.env` and write logs under `logs/`.


In [2]:
load_dotenv()
setup_logging()

<Logger voice_search (INFO)>

## Models

Load the speech, intent, and named-entity models, then build the searcher.


In [3]:
transcriber = Transcriber()
intent_detector = IntentDetector()
query_extractor = QueryExtractor(intent_detector)
searcher = Searcher(query_extractor)

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 17061.62it/s]


asr_freq: 16000


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.42it/s]
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 1271.39it/s]


## Checks

Run fixed phrases and compare the detected intent and searchable query.


In [4]:
cases = [
    ("Well tell me some news about Tom Cruise.", "news", "Tom Cruise"),
    ("What's up with Trump these days?", "news", "Trump"),
    ("Show me some t-shirts from Amazon.com", "ecommerce", "t-shirts"),
    ("get me some nyke shoes from amazon.com", "ecommerce", "nyke shoes"),
    ("find me some shoes similar to elon musk", "ecommerce", "shoes"),
    ("Tell me the latest headlines about Barack Obama.", "news", "Barack Obama"),
    ("What is happening in New York these days?", "news", "New York"),
    ("Any recent news about NASA?", "news", "NASA"),
    ("Find me wireless headphones from eBay.", "ecommerce", "wireless headphones"),
    ("Show me Adidas running shoes.", "ecommerce", "Adidas running shoes"),
    ("I want a jacket like the one Taylor Swift wore.", "ecommerce", "jacket"),
    ("What's the news about Apple's new iPhone?", "news", "Apple"),
    ("get me some addidas sneakers from flipkart", "ecommerce", "addidas sneakers"),
    ("Trump news", "news", "Trump"),
    ("looking for a cheap office chair", "ecommerce", "office chair"),
]

ok = 0
for text, actual_intent, actual_query in cases:
    record = VoiceSearch(user_query_text=text)
    intent_detector.detect_intent(record)
    query_extractor.extract_query(record)
    intent = record.intent
    query = record.searchable_query
    match = intent == actual_intent and query == actual_query
    ok += match
    print(text)
    print(f"{'ok' if match else 'WRONG'} ----- intent: {intent}, query: {query!r}")
    print()
print(f"ok: {ok}/{len(cases)}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 276.14it/s]


Well tell me some news about Tom Cruise.
ok ----- intent: news, query: 'Tom Cruise'



Batches: 100%|██████████| 1/1 [00:00<00:00, 339.87it/s]


What's up with Trump these days?
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 352.88it/s]


...

get me some addidas sneakers from flipkart
ok ----- intent: ecommerce, query: 'addidas sneakers'



Batches: 100%|██████████| 1/1 [00:00<00:00, 322.24it/s]


Trump news
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 373.62it/s]

looking for a cheap office chair
ok ----- intent: ecommerce, query: 'office chair'

ok: 15/15


## Demo

Open the microphone and file-upload UI on the models loaded above.


In [ ]:
demo = build_demo(VoiceSearchPipeline(transcriber, intent_detector, query_extractor, searcher))
demo.launch(share=True, server_port=7680)

* Running on local URL:  http://127.0.0.1:7680
* Running on public URL: https://329b6d84c901559a19.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Batches: 100%|██████████| 1/1 [00:00<00:00, 224.34it/s]


## Shut down

Shut down the demo server.


In [6]:
demo.close()

Closing server running on port: 7680
